<a href="https://colab.research.google.com/github/Amchuz/AI-ML_Personal_RoadMap/blob/main/week-01/day-01-and-02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# This is a guide for the LangChain Docs

URL: https://docs.langchain.com/oss/python/langchain/quickstart#build-a-real-world-agent

# Import any AI provider API KEY

Add your API key in the "key" symbol in the left side of colab and import it. I am using gemini free api key here

In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "LangChain_Day1"
os.environ["LANGSMITH_ENDPOINT"] = "https://apac.api.smith.langchain.com"
os.environ["LANGCHAIN_ENDPOINT"] = "https://apac.api.smith.langchain.com"  # add this

print("LANGSMITH_ENDPOINT:", os.environ["LANGSMITH_ENDPOINT"])
print("LANGCHAIN_ENDPOINT:", os.environ["LANGCHAIN_ENDPOINT"])
print("✅ All set")

# Start with a Simple Agent

The below is a setup to learn about agent and how a tool is used

In [ ]:
!pip install -U langchain-google-genai

In [ ]:
!pip install -U langsmith

In [ ]:
# pip install -qU langchain "langchain[google-genai]"
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content)

# Build a real-world agent

In the following example you will build a research agent that can answer questions about text files. Along the way you will explore the following concepts:

1. Detailed system prompts for better agent behavior
2. Create tools that integrate with external data
3. Model configuration for consistent responses
4. Conversational memory for chat-like interactions
5. Deep Agents for built-in features
6. Testing your agent

In [ ]:
SYSTEM_PROMPT = """You are a literary data assistant.

## Capabilities

- `fetch_text_from_url`: loads document text from a URL into the conversation.
Do not guess line counts or positions—ground them in tool results from the saved file."""

In [ ]:
pip install -U langchain deepagents

In [ ]:
import urllib.error
import urllib.request

from langchain.tools import tool

@tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL and return clean plain text."""
    import urllib.request, urllib.error
    from html.parser import HTMLParser

    class HTMLStripper(HTMLParser):
        def __init__(self):
            super().__init__()
            self.text = []
        def handle_data(self, data):
            self.text.append(data)
        def get_text(self):
            return "\n".join(self.text)

    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"

    text = raw.decode("utf-8", errors="replace")

    # Strip HTML if it looks like a webpage
    if "<html" in text.lower() or "<div" in text.lower():
        stripper = HTMLStripper()
        stripper.feed(text)
        text = stripper.get_text()

    return text

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gemini-2.5-flash-lite",
    model_provider="google-genai",
    temperature=0.5,
    timeout=600,
    max_tokens=25000,
    streaming=True,
)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [ ]:
from langsmith import traceable
from langchain.chat_models import init_chat_model

@traceable(project_name="LangChain_Day1")
def simple_llm_call(message: str) -> str:
    llm = init_chat_model(
        "gemini-2.5-flash-lite",
        model_provider="google-genai",
    )
    result = llm.invoke(message)
    return result.content

response = simple_llm_call("What is LangSmith in one sentence?")
print(response)

In [ ]:
from langchain.agents import create_agent
# from deepagents import create_deep_agent

agent = create_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

# deep_agent = create_deep_agent(
#     model=model,
#     tools=[fetch_text_from_url],
#     system_prompt=SYSTEM_PROMPT,
#     checkpointer=checkpointer,
# )

content = f"""
URL: https://raw.githubusercontent.com/langchain-ai/langchain/master/README.md

Answer:
1) How many lines contain the word 'agent'?
2) What is the first line that mentions 'LangSmith'?
3) One sentence summary of what this project is.
"""

agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "session-lc"}},
)
# deep_agent_result = deep_agent.invoke(
#     {"messages": [{"role": "user", "content": content}]},
#     config={"configurable": {"thread_id": "session-da"}},
# )
print("=" * 50)
print("LANGCHAIN AGENT RESULT")
print("=" * 50)
print(agent_result["messages"][-1].content)

# print("\n")

# print("=" * 50)
# print("DEEP AGENT RESULT")
# print("=" * 50)
# print(deep_agent_result["messages"][-1].content)

In [ ]:
# See every step the agents took
for i, msg in enumerate(agent_result["messages"]):
    print(f"[LC {i}] {type(msg).__name__}: {str(msg.content)[:80]}")

print("\n")

# for i, msg in enumerate(deep_agent_result["messages"]):
#     print(f"[DA {i}] {type(msg).__name__}: {str(msg.content)[:80]}")

In [ ]:
from langsmith import Client

client = Client()
projects = list(client.list_projects())
for p in projects:
    print(p.name)


In [ ]:
from langsmith import Client

client = Client()
print(client.api_url)  # print what URL it's actually using

In [ ]:
import langsmith
print(langsmith.__version__)